# W14-D3 实验 1：business-ontology.yaml 解析与校验——语义资产的机器体检

配套阅读：`第14周-Day3-ontology机器体检-语义资产半年后会变成什么.md`（md 是体检报告与裁决，本 notebook 是**可执行的体检过程**）：

1. **实验一 · 结构与完整性体检**：PyYAML 解析 + schema 形状校验（模块/子功能键集、意外键扫描）+ 完整性剖面（role/场景/术语按模块）；
2. **实验二 · 术语质量审计**：883 条术语条目的度分布、跨模块歧义、模块别名与他模块术语的重叠；
3. **实验三 · capability 锚点审计**：L 编号占位 / 命名命中 / 未命中三分 + 按模块命中率 + 跨模块复用；
4. **实验四 · 模块 × Context 程序化映射**：D2 手工 v0.1 映射编码为声明性输入 → 12×17 热力图 + 孤儿 Context + 机器可读体检报告（YAML 落盘）；
5. **实验五 · 熵增模拟**：PT-W4-D6 的六条装配规则 W1-W6 打在 ontology 上的判分（0/6）+ specs 增长下锚点覆盖率的两种命运曲线——回答 Today's Question「语义资产如果机器不可校验，半年后会变成什么」。

结论一句话：**这个文件的结构纪律是好的（无一处意外键），烂掉的方式不是"写乱"，而是"没人看"——完整性、锚点、溯源三处熵在无声增长。**

## 实验一：结构与完整性体检

先看两件 D2 只用眼睛看、今天用机器确认的事：
- **治理头**：文件顶层键只有 `modules:` ——没有版本、状态、维护人、变更流程（对比 BCM 的 frontmatter + ORE-1）；
- **schema 形状**：子功能允许的键 = `capabilities / role / scenarios / terms`，场景允许的键 = `name / description / source`。扫描全部 12 模块 102 子功能，统计违规。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np
import yaml, collections, os, hashlib, datetime

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

ONTO = "/root/docs/lanlnk/config/ontology/business-ontology.yaml"
SPECS_DIR = "/root/lnkcre/openspec/specs"
OUT_DIR = "/root/learning-notebooks/第14周"

raw = open(ONTO, encoding="utf-8").read()
data = yaml.safe_load(raw)
print("字体:", font_name)
print("ontology 行数:", len(raw.splitlines()), "| sha256:", hashlib.sha256(raw.encode()).hexdigest()[:16])

In [ ]:
mods = data["modules"]
print("① 治理检查：文件顶层键 =", list(data.keys()), "→ 无 frontmatter（版本/状态/维护人全缺）\n")

SF_KEYS = {"capabilities", "role", "scenarios", "terms"}
SC_KEYS = {"name", "description", "source"}
violations = []
rows = []
term_entries = 0
for m, md in mods.items():
    sfs = md.get("sub_functions", {})
    n_role = sum(1 for d in sfs.values() if "role" in d)
    n_scn = sum(len(d.get("scenarios", [])) for d in sfs.values())
    n_terms = sum(len(d.get("terms", [])) for d in sfs.values())
    term_entries += n_terms
    rows.append([m, len(sfs), n_role, n_scn, n_terms])
    for sf, d in sfs.items():
        violations += [f"{m}.{sf}: 意外键 {k}" for k in d if k not in SF_KEYS]
        for sc in d.get("scenarios", []):
            violations += [f"{m}.{sf}.scenario: 意外键 {k}" for k in sc if k not in SC_KEYS]

print(f"② schema 形状扫描：{len(mods)} 模块 / {sum(r[1] for r in rows)} 子功能")
print(f"   意外键违规 = {len(violations)}  → 形状纪律 102/102 规整\n")
print(f"③ 完整性剖面（{'模块':<10}{'子功能':>5}{'role':>6}{'场景':>5}{'术语':>5}）")
for r in rows:
    print(f"   {r[0]:<10}{r[1]:>5}{r[2]:>6}{r[3]:>5}{r[4]:>5}")
tot = [sum(r[i] for r in rows) for i in range(1, 5)]
print(f"   {'合计':<10}{tot[0]:>5}{tot[1]:>6}{tot[2]:>5}{tot[3]:>5}")
print(f"\n④ 结论：role 覆盖 {tot[1]}/{tot[0]}，场景覆盖 {tot[2]}/{tot[0]}"
      f"（且全部集中在资源管理 7 个子功能）→ 尾部烂尾是结构性的")

## 实验二：术语质量审计

D2 说"883 术语"是**条目数**。今天拆开看：同一条术语可以在多个子功能里登记（这是设计允许的——术语本来就跨场景），但**跨模块复用如果没有消歧声明，喂给 LLM 就是歧义**。重点指标：
- 度分布（一个术语被登记在几个子功能）；
- 跨模块歧义术语（同一个词在两个模块各有含义）；
- 模块级 aliases 与其他模块 terms 的重叠（别名撞术语）。

In [ ]:
term_loc = collections.defaultdict(list)
mod_alias = {}
for m, md in mods.items():
    mod_alias[m] = md.get("aliases", [])
    for sf, d in md["sub_functions"].items():
        for t in d.get("terms", []):
            term_loc[t].append(f"{m}.{sf}")

uniq = len(term_loc)
deg = collections.Counter(len(v) for v in term_loc.values())
multi = {t: v for t, v in term_loc.items() if len(v) > 1}
cross_mod = {t: v for t, v in term_loc.items() if len({loc.split(".")[0] for loc in v}) > 1}

print(f"术语条目 {term_entries} → 唯一术语 {uniq}（{term_entries - uniq} 条为复用登记）")
print(f"度分布（术语被登记在几个子功能）: {dict(sorted(deg.items()))}")
print(f"多处出现: {len(multi)} 个 | 跨模块歧义: {len(cross_mod)} 个（占唯一术语 {len(cross_mod)/uniq:.1%}）\n")
print("复用度 TOP5（喂 LLM 前最该消歧的词）:")
for t, v in sorted(multi.items(), key=lambda x: -len(x[1]))[:5]:
    mods_of = collections.Counter(loc.split(".")[0] for loc in v)
    print(f"  「{t}」× {len(v)}  → " + ", ".join(f"{k}×{n}" for k, n in mods_of.items()))

alias_hit = []
for m, als in mod_alias.items():
    for a in als:
        other = [loc for loc in term_loc.get(a, []) if not loc.startswith(m)]
        if other:
            alias_hit.append((a, m, other[:2]))
print(f"\n模块别名与他模块术语重叠: {len(alias_hit)} 处")
for a, m, o in alias_hit[:6]:
    print(f"  别名「{a}」（{m}）↔ 术语出现在 {o}")

## 实验三：capability 锚点审计

D2 已给出全局数字（23 L 占位 / 79 命名中 33 命中 spec = 42%）。今天按模块拆开——**贴纸质量是极不均匀的**：
财务管理 12/12 全命中，而移动端 1/15、数据决策 2/14、预算管理 1/8。跨模块复用标签（platform-foundation 挂 7 个模块 33 个子功能）是"万能贴纸"的量化证据。

In [ ]:
cap_sf = collections.Counter()
cap_mods = collections.defaultdict(set)
for m, md in mods.items():
    for sf, d in md["sub_functions"].items():
        for c in d.get("capabilities", []):
            cap_sf[c] += 1
            cap_mods[c].add(m)

L = {c for c in cap_sf if c.startswith("L") and c[1:].isdigit()}
specs = set(os.listdir(SPECS_DIR))
named = [c for c in cap_sf if c not in L]
hits = [c for c in named if c in specs]
cross = {c: len(ms) for c, ms in cap_mods.items() if len(ms) > 1}

print(f"capability 标签 {len(cap_sf)}：L 编号占位 {len(L)} + 命名 {len(named)}"
      f"（其中命中 openspec spec 目录 {len(hits)} = {len(hits)/len(named):.0%}）")
print(f"反向覆盖：{len(hits)}/{len(specs)} specs 被引用 = {len(hits)/len(specs):.0%}")
print(f"跨模块复用标签 {len(cross)} 个，TOP5: " +
      ", ".join(f"{c}×{n}模块" for c, n in sorted(cross.items(), key=lambda x: -x[1])[:5]) +
      f"；platform-foundation 实挂 {cap_sf['platform-foundation']} 个子功能\n")

print(f"{'模块':<12}{'标签':>4}{'命名':>5}{'命中':>4}{'命中率':>6}")
per_mod = []
for m, md in mods.items():
    cs = set()
    for d in md["sub_functions"].values():
        cs |= set(d.get("capabilities", []))
    nm = [c for c in cs if c not in L]
    ht = sum(1 for c in nm if c in specs)
    per_mod.append((m, len(cs), len(nm), ht))
    print(f"{m:<12}{len(cs):>4}{len(nm):>5}{ht:>4}{(ht/len(nm) if nm else float('nan')):>7.0%}")
print("\n贴纸质量极不均匀：财务 12/12 全锚定 vs 移动端 1/15、数据决策 2/14、预算管理 1/8")

## 实验四：模块 × Context 程序化映射（D2 手工 v0.1 的机器版）

D2 的手工映射表今天编码为**声明性输入**（`MODULE_TO_CONTEXTS`，依据 Domain Model §2 Coverage Matrix + Crosswalk），程序负责：算扇出、触达率、找孤儿 Context、并和实验三的 spec 证据并排——**映射不是文档装饰，是查询路由的必需品**。产物落盘为机器可读 YAML（D6 定稿包的直接原料）。

In [ ]:
MODULE_TO_CONTEXTS = {  # D2 手工校准 v0.1（声明性输入：Domain Model §2 + Crosswalk 依据）
    "资源管理": [1, 5], "招商管理": [3], "合同管理": [4],
    "财务管理": [6, 7, 8, 9], "运营管理": [10, 13], "物业管理": [11],
    "推广营销": [14], "系统管理": [], "资产管理（系统对接）": [],
    "移动端": [], "数据决策": [17], "预算管理": [6],
}
CTX = {1: "01 Asset Fnd", 2: "02 Party Core", 3: "03 Leasing", 4: "04 Contract",
       5: "05 Lease/Occ", 6: "06 Billing", 7: "07 Collection", 8: "08 TaxInvoice",
       9: "09 AcctBridge", 10: "10 Operations", 11: "11 Property", 12: "12 Engineering",
       13: "13 WorkOrder", 14: "14 Marketing", 15: "15 Cust/Member", 16: "16 Parking",
       17: "17 BI/Analytics"}
mod_names = list(mods.keys())
M = np.zeros((len(mod_names), 17))
for i, m in enumerate(mod_names):
    for c in MODULE_TO_CONTEXTS.get(m, []):
        M[i, c - 1] = 1
touched = {c + 1 for i in range(len(mod_names)) for c in range(17) if M[i, c]}
orphan = sorted(set(range(1, 18)) - touched)
fanout = sum(len(v) for v in MODULE_TO_CONTEXTS.values())
print(f"扇出合计 {fanout}（12 模块 → {len(touched)}/17 Context，平均 {fanout/len(mods):.1f}）")
print(f"孤儿 Context（ontology 模块层无入口）: " + ", ".join(f"{CTX[c]}({c})" for c in orphan))
print("其中 02 Party Core 恰是 Gap Analysis 最重整改对象（三业态 Party 未建）→ 盲区与债务重合")

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(M, cmap="YlGn", vmin=0, vmax=1.2, aspect="auto")
ax.set_xticks(range(17)); ax.set_xticklabels([CTX[c] for c in range(1, 18)], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(mod_names))); ax.set_yticklabels(mod_names, fontsize=9)
hit_by_mod = {m: h for m, _, _, h in per_mod}
for i, m in enumerate(mod_names):
    for c in MODULE_TO_CONTEXTS.get(m, []):
        ax.text(c - 1, i, "●", ha="center", va="center", color="#1a5276")
    h = hit_by_mod[m]
    ax.text(17.1, i, f"spec锚定 {h}", va="center", fontsize=8, color="#555")
ax.set_title("模块 × Context 映射（程序化 v0.1，●=映射）｜右侧=该模块 capability 命中 spec 数", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/w14d3_模块Context映射.png", dpi=150, bbox_inches="tight"); plt.show()

report = {
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "source": {"file": ONTO, "sha256_16": hashlib.sha256(raw.encode()).hexdigest()[:16]},
    "counts": {"modules": len(mods), "sub_functions": tot[0], "scenario_entries": tot[2],
               "term_entries": term_entries, "unique_terms": uniq, "capability_labels": len(cap_sf)},
    "findings": {
        "governance_frontmatter": False,
        "schema_shape_violations": len(violations),
        "role_coverage": f"{tot[1]}/{tot[0]}",
        "scenario_coverage": f"{tot[2]}/{tot[0]} (全部在资源管理)",
        "terms_multi_location": len(multi), "terms_cross_module": len(cross_mod),
        "capability_L_placeholder": len(L), "capability_named": len(named),
        "capability_spec_hits": len(hits), "capability_hit_rate": f"{len(hits)/len(named):.0%}",
        "spec_reverse_coverage": f"{len(hits)}/{len(specs)} = {len(hits)/len(specs):.0%}",
        "orphan_contexts": [f"{c} {CTX[c]}" for c in orphan],
    },
    "module_to_contexts": {m: [CTX[c] for c in cs] for m, cs in MODULE_TO_CONTEXTS.items()},
}
rp = f"{OUT_DIR}/w14d3-ontology-health-report.yaml"
yaml.safe_dump(report, open(rp, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print("机器可读体检报告已落盘:", rp)

## 实验五：熵增模拟——半年后会变成什么？

两个动作：
1. **W1-W6 判分**：把 PT-W4-D6 的六条装配规则（谓词可解析/挂载点存在/effects∈冻结注册表/依赖可解析/边界包含/变体锚定）逐条对 ontology 求值——预期 0/6，全部因**构件缺失**而无法执行。这是 D2"缺构件"结论的机器判定版。
2. **两种命运曲线**：lnkcre 的 specs 在涨（8 月内 261→273）。假设每月 +10 个新 spec：
   - **无校验线**：ontology 无人维护，被引用数固定 33 → 锚点覆盖率 33/(273+10k) 单调稀释；
   - **有 CI 线**：每个新 spec 触发校验并回填标签 → 覆盖率随增长而抬升。
   这是情景模型（假设显式标注），不是测量——但两条曲线的**方向差异**不依赖参数取值。

In [ ]:
# PT-W4-D6 六条装配规则 × ontology：构件缺失判定
six = [
    ("W1", "Rule 谓词名词 ∈ Entity 定义", "无 Entity/Rule 构件", "N/A-fail"),
    ("W2", "Guard 挂载点 ∈ 已声明迁移", "无 Lifecycle 构件", "N/A-fail"),
    ("W3", "effects ∈ effect-registry 冻结 5 类", "无 Event 构件", "N/A-fail"),
    ("W4", "capability_dependencies ∈ Capability Map", "无依赖/状态构件", "N/A-fail"),
    ("W5", "Agent 认知边界 ⊆ 17 Context", "无 Agent Card 构件", "N/A-fail"),
    ("W6", "Variant 规则锚定 base Rule Card", "无 Rule Card 构件", "N/A-fail"),
]
print("六条装配规则判分（0/6 通过）——词汇层喂不饱语义消费方：")
for w, rule, why, verdict in six:
    print(f"  {w}  {rule:<28} 缺失原因: {why:<14} → {verdict}")

# 熵增模拟：specs 增长下锚点覆盖率的两种命运（情景模型，假设已显式标注）
months = np.arange(0, 7)
growth = 10  # 假设：每月 +10 spec（8 月内 261→273 的量级）
no_ci = 33 / (273 + growth * months)                    # 无人维护：被引用数冻结在 33
with_ci = (33 + growth * months) / (273 + growth * months)  # CI 线：每个新 spec 校验并回填
cross_now = len(cross_mod) / uniq                       # 跨模块歧义率基线 4.7%
cross_growth = cross_now * (1 + 40 * months / uniq)  # 歧义术语按规模线性外推

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(months, no_ci * 100, "o-", color="#c0392b", label="无校验线：ontology 冻结")
axes[0].plot(months, with_ci * 100, "s-", color="#1e8449", label="CI 线：新 spec 强制回填")
axes[0].set_xlabel("月后"); axes[0].set_ylabel("spec 锚点覆盖率 (%)")
axes[0].set_title(f"半年后：{no_ci[0]:.0%} → 无校验 {no_ci[-1]:.0%} vs 有校验 {with_ci[-1]:.0%}")
axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(months, cross_growth * 100, "^-", color="#8e44ad")
axes[1].set_xlabel("月后"); axes[1].set_ylabel("跨模块歧义术语数（按线性外推）")
axes[1].set_title(f"术语歧义熵：{len(cross_mod)} → {cross_growth[-1]:.0f} 个（无人发现=无人消歧）")
axes[1].grid(alpha=.3)
fig.suptitle("熵增模拟（情景假设：+10 spec/月、+40 术语/月）——两条曲线的方向差异不依赖参数", fontsize=10)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/w14d3_熵增模拟.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"半年锚点覆盖率：无校验 12%→{no_ci[-1]:.0%}（稀释），有校验→{with_ci[-1]:.0%}（抬升）")

## 结论

1. **烂的方式不是写乱，是没人看**：schema 形状 102/102 规整、零意外键；但 role 7/102、场景 15/102（全在资源管理）、治理头为零——半成品 + 无基线，熵在三个维度无声增长（完整性/歧义/锚点）。
2. **机器可校验是 Semantic Model 的存在理由**：今天这套体检（~150 行 Python）在无 frontmatter 的文件上依然跑出了全部指标——证明对账层不需要改造源文件就能度量它；W1-W6 的 0/6 则说明词汇层喂不饱语义消费方。
3. **孤儿 Context = ontology 的盲区清单**：02 Party Core / 12 Engineering / 15 Customer-Member / 16 Parking 在模块层无入口——其中 Party Core 恰是 Gap Analysis 里最重的整改对象（三业态 Party 未建）。